In [10]:
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import joblib
import gc
def descargar_clima_historico(fecha_inicio, fecha_fin):
    """Descarga el clima histórico de Valencia para el rango de fechas dado."""
    lat = 39.4699
    lon = -0.3763
    start_str = fecha_inicio.strftime('%Y-%m-%d')
    end_str = fecha_fin.strftime('%Y-%m-%d')
    
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_str}&end_date={end_str}&hourly=temperature_2m,precipitation&timezone=Europe%2FMadrid"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        df_clima = pd.DataFrame({
            "fecha_hora_redondeada": pd.to_datetime(data["hourly"]["time"]),
            "temperatura": data["hourly"]["temperature_2m"],
            "precipitacion": data["hourly"]["precipitation"]
        })
        df_clima["precipitacion"] = df_clima["precipitacion"].fillna(0)
        df_clima["temperatura"] = df_clima["temperatura"].ffill()
        
        return df_clima
    else:
        raise Exception(f"Error al descargar clima: {response.status_code}")
print("🚲 Cargando datos de Valenbisi...")
df = pd.read_csv("valenbici_limpio.csv")
df["fecha"] = pd.to_datetime(df["fecha_actualizacion_final"], dayfirst=True)

df1 = df[df["fecha"] >= df["fecha"].max() - pd.DateOffset(years=1)].copy()

🚲 Cargando datos de Valenbisi...


C:\Users\xboxp\AppData\Local\Temp\ipykernel_5492\3394672738.py:45: DtypeWarning: Columns (0: name, 1: globalid, 2: update_jcd) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("valenbici_limpio.csv")


In [11]:

df1["fecha_hora_redondeada"] = df1["fecha"].dt.floor('h')

fecha_min = df1["fecha"].min()
fecha_max = df1["fecha"].max()

df_clima = descargar_clima_historico(fecha_min, fecha_max)

print("Cruzando datos de bicis con datos climáticos...")
df1 = pd.merge(df1, df_clima, on="fecha_hora_redondeada", how="left")

df1["fecha_dia"] = df1["fecha"].dt.date 
df1["hora"] = df1["fecha"].dt.hour
df1["dia_semana"] = df1["fecha"].dt.dayofweek
df1["mes"] = df1["fecha"].dt.month

print("Calculando promedios históricos (Baseline)...")

df_historico = df1.groupby(["numero", "hora", "dia_semana", "mes"]).agg({
    "bicis_disponibles": "mean",
    "espacios_libres": "mean"
}).reset_index()

df_historico.rename(columns={
    "bicis_disponibles": "media_hist_bicis",
    "espacios_libres": "media_hist_huecos"
}, inplace=True)

print("Calculando datos diarios reales con clima...")
df_diario = df1.groupby(["numero", "fecha_dia", "hora", "dia_semana", "mes"]).agg({
    "bicis_disponibles": "mean", 
    "espacios_libres": "mean",
    "temperatura": "mean",
    "precipitacion": "sum"
}).reset_index()

print("Uniendo el histórico con el diario...")
df_final = pd.merge(df_diario, df_historico, on=["numero", "hora", "dia_semana", "mes"], how="left")


df_historico.to_csv("promedios_historicos.csv", index=False)
print("¡Promedios exportados correctamente!")


del df, df1, df_clima, df_historico, df_diario
gc.collect()

features_b = ["numero", "hora", "dia_semana", "mes", "temperatura", "precipitacion", "media_hist_bicis"]
features_h = ["numero", "hora", "dia_semana", "mes", "temperatura", "precipitacion", "media_hist_huecos"]

X_b = df_final[features_b]
X_h = df_final[features_h]

y_bicis = df_final["bicis_disponibles"]
y_huecos = df_final["espacios_libres"]


X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_bicis, test_size=0.2, random_state=42)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_huecos, test_size=0.2, random_state=42)

print("Entrenando modelo de BICICLETAS...")
model_bicis = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=10, random_state=42, n_jobs=-1)
model_bicis.fit(X_train_b, y_train_b)

preds_b = model_bicis.predict(X_test_b)
print(f"--- RESULTADOS BICIS ---")
print(f"MAE: {mean_absolute_error(y_test_b, preds_b):.2f} bicicletas")
print(f"R² Score: {r2_score(y_test_b, preds_b):.2f}\n")

print("Entrenando modelo de HUECOS...")
model_huecos = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=10, random_state=42, n_jobs=-1)
model_huecos.fit(X_train_h, y_train_h)

preds_h = model_huecos.predict(X_test_h)
print(f"--- RESULTADOS HUECOS ---")
print(f"MAE: {mean_absolute_error(y_test_h, preds_h):.2f} huecos libres")
print(f"R² Score: {r2_score(y_test_h, preds_h):.2f}\n")

joblib.dump(model_bicis, "model_bicis.pkl")
joblib.dump(model_huecos, "model_huecos.pkl")

print("¡Modelos y promedios exportados correctamente!")

🌦️ Descargando datos climáticos desde 2024-09-23 hasta 2025-09-23...
🔗 Cruzando datos de bicis con datos climáticos...
📊 Calculando promedios históricos (Baseline)...
📊 Calculando datos diarios reales con clima...
🔗 Uniendo el histórico con el diario...
✅ ¡Promedios exportados correctamente!
⚙️ Entrenando modelo de BICICLETAS...
--- RESULTADOS BICIS ---
MAE: 3.02 bicicletas
R² Score: 0.61

⚙️ Entrenando modelo de HUECOS...
--- RESULTADOS HUECOS ---
MAE: 3.07 huecos libres
R² Score: 0.71

✅ ¡Modelos y promedios exportados correctamente!
